# Chapter 4 · Scope contracts

The agent of chapter 2, with its rules checked on editing and at the end, now receives the contract of each task: goal, paths it may change, forbidden paths and acceptance criteria. Three variants are compared: no contract, the contract in the request, and the contract checked on editing and at the gate.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 4: scope contracts."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with greedy decoding. Returns the text and the
    input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_TOKENS_STEP,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefill_chunk_size=1024,
                                pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. The rules of chapter 2

`check_rules` has a new parameter, `require_test`, for when the contract forbids touching `tests/`.

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name


def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer, require_test=True,
                allows_docs=None):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer. With "require_test" False,
    R7 does not ask for a new check; "allows_docs", if given, decides R3 instead of
    the request."""
    if allows_docs is None:
        allows_docs = bool(ASKS_DOCS.search(task["request"]))
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not allows_docs:
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test or not require_test,
             "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not self.allows_docs():
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists:
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def allows_docs(self):
        """True if the request asks to change the terms and conditions (R3)."""
        return bool(ASKS_DOCS.search(self.task["request"]))

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result

## 4. The contract

The contract of a task, as the agent sees it, and the checks, which can be tried without the model.

In [ ]:
def contract_text(contract):
    """A contract (like data/tasks/Txx/contract.json) written for the agent: goal,
    paths it may change, forbidden paths and numbered acceptance criteria (C1,
    C2…)."""
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    return ("Task contract\n"
            f"Goal: {contract['goal']}\n"
            f"You may change: {', '.join(contract['allowed_paths'])}\n"
            f"Don't touch: {', '.join(contract['forbidden_paths'])}\n"
            f"Acceptance criteria:\n{criteria}\n"
            "The contract overrides AGENTS.md: if it forbids tests/, don't add tests.")


def forbids_tests(contract):
    """True if the contract forbids changing tests/ (for example, a refactoring
    that must pass the tests without touching them)."""
    return rt.path_matches("tests/x.py", contract["forbidden_paths"])


def check_contract(contract, changes, answer):
    """What breaks the contract at the end: files outside the allowed paths or in
    the forbidden ones, and criteria the answer does not explain how it meets."""
    broken = {}
    paths = rt.check_scope(changes["changed"], contract)
    outside = sorted(set(paths["out_of_scope"]) | set(paths["forbidden"]))
    if outside:
        broken["Scope"] = (f"you changed {', '.join(outside)}, which the contract "
                           "does not allow")
    numbers = range(1, len(contract["acceptance_criteria"]) + 1)
    unexplained = [f"C{i}" for i in numbers
                   if not re.search(rf"\bC{i}\b", answer or "")]
    if unexplained:
        broken["Criteria"] = ("your answer does not say how it meets "
                              + ", ".join(unexplained)
                              + " (cite each criterion by its number)")
    return broken


class ToolsWithContract(ToolsWithRules):
    """The tools with the rules of chapter 2 and, on top of them, the contract:
    nothing can be edited or created outside the allowed paths."""

    def __init__(self, tools, task, contract):
        super().__init__(tools, task)
        self.contract = contract

    def allows_docs(self):
        """The contract overrides R3: docs/ can be changed if the contract allows it."""
        return rt.path_matches("docs/terms_and_conditions.md",
                               self.contract["allowed_paths"])

    def broken_rule(self, name, path):
        """The rule or the part of the contract the edit would break, or None."""
        contract = self.contract
        if rt.path_matches(path, contract["forbidden_paths"]):
            return f"Contract: {path} is among the paths you must not touch."
        if not rt.path_matches(path, contract["allowed_paths"]):
            return (f"Contract: {path} is not among the paths you may change ("
                    + ", ".join(contract["allowed_paths"]) + ").")
        return super().broken_rule(name, path)

In [ ]:
task = rt.read_task("T17")
print(contract_text(task["contract"]))
with rt.Workspace() as workspace:
    t = ToolsWithContract(rt.Tools(workspace), task, task["contract"])
    print(t.call("edit", {"path": "shipping.py", "old": "a", "new": "b"}))
    print(check_contract(task["contract"], rt.differences(workspace.source, workspace.folder),
                         "DONE: C1 and C2 are met."))

## 5. The agent

The task of the new tune-up price, without a contract and with the checked contract. Look at the first file the agent reads.

In [ ]:
def agent(tok, model, task, contract=None, scope=None, max_steps=MAX_STEPS,
          verbose=False):
    """The agent of chapter 2 (rules checked on editing and at the end) with a
    contract: without it (None), "visible" in the request, or "checked", visible and
    also enforced on editing and at the gate. The contract is the task's, unless
    another one is passed in "scope". Returns the row."""
    start = time.time()
    scope = scope or task["contract"]
    request = task["request"]
    if contract:
        request += "\n\n" + contract_text(scope)
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": request}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    with rt.Workspace() as workspace:
        base = rt.Tools(workspace)
        if contract == "checked":
            tools = ToolsWithContract(base, task, scope)
        else:
            tools = ToolsWithRules(base, task)
        for steps in range(1, max_steps + 1):
            text, trimmed = trim_conversation(tok, messages)
            if trimmed is None:
                break
            answer, n_input, n_output = generate(tok, model, text)
            n_in, n_out = n_in + n_input, n_out + n_output
            calls = read_calls(answer)
            if not calls:
                messages.append({"role": "assistant", "content": answer})
                if verbose:
                    print(f"[{steps}] {answer}")
                if says_done(answer) and rejections < MAX_REJECTIONS:
                    changes = rt.differences(workspace.source, workspace.folder)
                    broken = check_rules(
                        task, base.log, changes, answer,
                        require_test=not (contract and forbids_tests(scope)),
                        allows_docs=(tools.allows_docs()
                                     if contract == "checked" else None))
                    if contract == "checked":
                        broken.update(check_contract(scope, changes, answer))
                    if broken:
                        rejections += 1
                        notice = ("It isn't done yet: "
                                  + "; ".join(f"{r}: {t}"
                                              for r, t in sorted(broken.items()))
                                  + ". Keep working.")
                        messages.append({"role": "user", "content": notice})
                        if verbose:
                            print(f"    GATE: {notice}")
                        continue
                final = answer
                break
            thought = answer.split("<tool_call>")[0].strip()
            messages.append({"role": "assistant", "content": thought,
                             "tool_calls": [{"type": "function", "function": c}
                                            for c in calls if c["name"]]})
            for call in calls:
                if call["name"] is None:
                    result = {"error": "The tool call cannot be understood."}
                else:
                    result = tools.call(call["name"], call["arguments"])
                if verbose:
                    print(f"[{steps}] {call['name']}"
                          f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
                messages.append({"role": "tool", "name": call["name"] or "?",
                                 "content": json.dumps(result, ensure_ascii=False)})
        evaluation = rt.evaluate_attempt(workspace, task)
    done = final is not None and says_done(final)
    return {"task": task["id"], "type": task["meta"]["type"],
            "solved": evaluation["solved"], "says_done": done,
            "false_done": done and not evaluation["solved"],
            "out_of_scope": evaluation["out_of_scope"],
            "forbidden": evaluation["forbidden"], "rejections": rejections,
            "hidden": evaluation["hidden"], "visible": evaluation["visible"],
            "steps": steps, "input_tokens": n_in, "output_tokens": n_out,
            "seconds": round(time.time() - start, 1),
            "answer": final or "(no final answer)",
            "calls": [[r["tool"], r["arguments"].get("path", ""), r["ok"]]
                      for r in base.log],
            "diff": evaluation["diff"]}


VARIANTS = {
    "no contract": {},
    "visible contract": {"contract": "visible"},
    "checked contract": {"contract": "checked"},
}

In [ ]:
for name in ("no contract", "checked contract"):
    print(f"--- {name}")
    row = agent(tok, model, rt.read_task("T17"), verbose=True, **VARIANTS[name])
    print("Solved:", row["solved"], "· Out of scope:", row["out_of_scope"])

## 6. The fourteen tasks with the three variants

By default a sample is run; with `COMPLETE = True`, the book's whole experiment.

In [ ]:
def summary(rows):
    """The totals of a variant."""
    return {"tasks": len(rows),
            "solved": sum(r["solved"] for r in rows),
            "says_done": sum(r["says_done"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "with_out_of_scope": sum(bool(r["out_of_scope"]) for r in rows),
            "files_outside": sum(len(r["out_of_scope"]) for r in rows),
            "hidden": sum(r["hidden"] for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, options, tasks=TASKS):
    """One variant with every task: its rows and its summary."""
    rows = []
    for task_id in tasks:
        row = agent(tok, model, rt.read_task(task_id), **options)
        rows.append(row)
        print(f"  {task_id} {'solved' if row['solved'] else 'no    '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  outside={len(row['out_of_scope'])}"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False two tasks are run (about an hour on Colab's T4);
# with True, the chapter's whole experiment (about fifteen hours on the T4).
COMPLETE = False
tasks = TASKS if COMPLETE else ["T04", "T17"]
results = {}
for name, options in VARIANTS.items():
    print(f"== {name}")
    results[name] = run_all(tok, model, options, tasks=tasks)
    print(results[name]["summary"])

The book's full results, with all the tasks, are saved in `data/results/chapter_4/`, which `prepare` has downloaded:

In [ ]:
import json
saved = json.loads((ROOT / "data" / "results" / "chapter_4" / "results.json").read_text(encoding="utf-8"))
for name, r in saved.items():
    print(name, r["summary"])

## Exercises

The results files the exercises cite are in `data/results/chapter_4/`, which `prepare` downloads with the data.

1. Write the contract of the sold-out task (T05) without looking at its own, using only the request. Then compare it with `data/tasks/T05/contract.json`. What did you miss? What did you add that was not needed? *(No GPU.)*

2. In the task of the new price, the contract puts `config.py` first among the allowed paths. Change the order and put it last. Does the first file the agent reads change? And the result? *(GPU, about 10 minutes.)*

3. Use a checked contract with the paths but without acceptance criteria. Which part of the improvement comes from the paths and which part from the criteria? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T10, T17 and T19. *(No GPU with the saved results; GPU, about an hour, with those tasks.)*

4. Look in the task bank for a contract with a criterion that cannot be checked with an automatic test. Rewrite it so that it can. *(No GPU.)*

5. The checked contract rejects any path that is not allowed when editing. Does it also reject reading it? Look at the code of `ToolsWithContract` and say which tools it watches and which ones it doesn't. *(No GPU.)*

6. Ask the agent to write the contract itself before starting, from the request and the list of files, and use it as a checked contract. Does it solve as many tasks as with the contract we wrote? Compare one of its contracts with ours. The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T10, T17 and T19. *(No GPU with the saved results; GPU, about 30 minutes, with those tasks.)*

7. The gate requires the answer to cite each criterion by its number, but it does not check that it is met. For the task of the new price, write a function that really checks the three criteria on the copy of the repository, importing `estimates` and `notices` and reading the terms. *(No GPU.)*

8. What happens if the contract is wrong, for example if it leaves out a necessary path? Remove `config.py` from the allowed paths of the task of the new price and run it with the checked contract. What does the agent do when its edit is rejected? *(GPU, about 10 minutes.)*

9. Compute the cost per solved task of the three variants, in input tokens and in seconds. Is the checked contract worth it? *(No GPU.)*

10. **Project.** Write a contract for each of three real tasks of a repository of yours, with a goal, paths and acceptance criteria. Give them to a coding agent, a commercial one or the book's, and note in which tasks the contract helped it and in which it got in its way.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). The definition of done, in the [Scrum Guide](https://scrumguides.org/scrum-guide.html). The North Cycles store, its program, its tasks and its contracts are fictional. Original code and experiments.